# ML Laboratuvarı · Modül 5, Ünite 9: Boosting ve SHAP

**Prof. Dr. Mehmet Songur** · ML Laboratuvarı, Modül 5

Veri: Tayvan'da bir bankanın 30.000 kredi kartı müşterisi (UCI, Yeh ve Lien 2009, CC BY 4.0). Kurulum hücresi `kredi` adlı tabloyu indirir ve gerekirse `shap` kütüphanesini kurar. Hedef `default`: Ekim 2005'te temerrüt (1). Ağaçlar ödeme kodlarını (`pay_0` … `pay_6`) ham hâlleriyle kullanabilir; korunan özellikler (`sex`, `marriage`) dışarıda kalır. Kodu **Gemini'ye yazdır**, sonra **denetle**.

**Kurallar**
1. **Kurulum** ve **Kontrol** hücrelerini değiştirme; yalnızca çalıştır.
2. Görev hücrelerinde kodu Gemini ile üret; çalıştırmadan önce oku.
3. Kontrol hücresi geçerse bir **tamamlama kodu** verir; onu ML Laboratuvarı'ndaki kutuya yapıştır.

In [ ]:
#@title Kurulum — çalıştır, değiştirme

import base64, subprocess, sys, warnings
try:
    import shap
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

kredi = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/credit/taiwan_default.csv")
print("kredi:", kredi.shape, "| temerrüt oranı:", round(kredi["default"].mean(), 3), "| shap", shap.__version__)
_E, _T = train_test_split(kredi.copy(), test_size=0.3, random_state=0, stratify=kredi["default"])
_SUTUN = [c for c in kredi.columns if c not in ("default", "sex", "marriage")]
_ADAY = [100, 200, 400, 800]

def _k(s):
    return base64.b64decode(s).decode()

def _hgb(n):
    return HistGradientBoostingClassifier(max_iter=n, learning_rate=0.05, max_leaf_nodes=8, early_stopping=False, random_state=1)

def kontrol_A(egitim, test, X_e, X_t, secilen_n, model, p_test, auc_test):
    if len(test) != len(_T) or set(test.index) != set(_T.index) or set(egitim.index) != set(_E.index):
        print("❌ Bölme beklenenden farklı: train_test_split(kredi, test_size=0.3, random_state=0, stratify=kredi['default'])."); return
    if {"sex", "marriage"} & set(X_e.columns):
        print("❌ Korunan özellikler (sex, marriage) modelde. Bunları dışarıda bırak."); return
    if "default" in X_e.columns:
        print("❌ Hedef ('default') özelliklerin içinde: hedef sızıntısı."); return
    if set(X_e.columns) != set(_SUTUN) or list(X_t.columns) != list(X_e.columns):
        print("❌ Özellikler beklenenden farklı: default, sex ve marriage dışındaki bütün ham sütunlar, eğitim ve testte aynı sırayla."); return
    if not isinstance(model, HistGradientBoostingClassifier):
        print("❌ model bir HistGradientBoostingClassifier olmalı."); return
    if model.early_stopping is not False:
        print("❌ early_stopping açık: model ağaç sayısını eğitim verisinden ayırdığı bir parçayla kendisi seçiyor. early_stopping=False yap."); return
    if model.learning_rate != 0.05 or model.max_leaf_nodes != 8:
        print("❌ Ayarlar beklenenden farklı: learning_rate=0.05, max_leaf_nodes=8."); return
    print("Kontrol: ağaç sayısı adayları eğitim verisinde çapraz doğrulanıyor (birkaç saniye)…")
    cv = {n: cross_val_score(_hgb(n), X_e, egitim["default"], cv=5, scoring="roc_auc").mean() for n in _ADAY}
    ref = max(cv, key=cv.get)
    if int(secilen_n) != ref:
        tst = {n: roc_auc_score(test["default"], _hgb(n).fit(X_e, egitim["default"]).predict_proba(X_t)[:, 1]) for n in _ADAY}
        if int(secilen_n) == max(tst, key=tst.get):
            print(f"❌ Ağaç sayısı ({secilen_n}) test verisinde seçilmiş. Seçimi eğitim verisinde cross_val_score(..., cv=5, scoring='roc_auc') ile yap."); return
        print(f"❌ Seçilen ağaç sayısı ({secilen_n}) beklenenden farklı. Adaylar {_ADAY}, eğitim verisinde 5 katlı çapraz doğrulama, ölçü roc_auc."); return
    if model.max_iter != int(secilen_n):
        print("❌ model, seçilen ağaç sayısıyla (max_iter=secilen_n) kurulmamış."); return
    p = np.asarray(p_test, dtype=float)
    if len(p) != len(test) or len(np.unique(p)) <= 2:
        print("❌ p_test olasılık değil. predict yerine predict_proba(...)[:, 1] kullan."); return
    if not np.allclose(p, model.predict_proba(X_t)[:, 1]):
        print("❌ p_test, modelin X_t için verdiği temerrüt olasılıklarıyla uyuşmuyor."); return
    a = roc_auc_score(test["default"], p)
    if abs(float(auc_test) - a) > 1e-6:
        print("❌ auc_test, roc_auc_score(test['default'], p_test) ile uyuşmuyor."); return
    print(f"✅ Tüm kontroller geçti (ağaç sayısı {ref}, test AUC ≈ {a:.3f}; Modül 4'ün lojistik regresyonu 0,762). Tamamlama kodun: " + _k("TUw1LUJTVC02UVJN"))

def kontrol_B(model, X_t, sv, taban, onem):
    Xs = X_t.iloc[:2000]
    sv = np.asarray(sv)
    if sv.shape != Xs.shape:
        print(f"❌ sv'nin boyutu {sv.shape}; ilk 2.000 test müşterisi için {Xs.shape} olmalı (X_t.iloc[:2000])."); return
    fark = np.abs(float(taban) + sv.sum(1) - model.decision_function(Xs)).max()
    if fark > 1e-6:
        print(f"❌ Toplama özelliği tutmuyor (en büyük fark {fark:.2g}): taban + sv.sum(1), model.decision_function(X_t.iloc[:2000]) ile aynı olmalı. taban için shap.TreeExplainer(model).expected_value kullan."); return
    ref = shap.TreeExplainer(model).shap_values(Xs)
    if not np.allclose(sv, ref, atol=1e-6):
        print("❌ sv, shap.TreeExplainer(model).shap_values(X_t.iloc[:2000]) ile uyuşmuyor."); return
    if not isinstance(onem, pd.Series) or set(onem.index) != set(Xs.columns):
        print("❌ onem, indeksinde sütun adları olan bir pandas Series olmalı."); return
    beklenen = pd.Series(np.abs(sv).mean(0), index=Xs.columns)
    if not np.allclose(onem.reindex(beklenen.index).values, beklenen.values):
        print("❌ onem, her sütunun ortalama |SHAP| değeri olmalı: np.abs(sv).mean(axis=0)."); return
    if not onem.is_monotonic_decreasing:
        print("❌ onem büyükten küçüğe sıralı olmalı (sort_values(ascending=False))."); return
    print(f"✅ Tüm kontroller geçti (en önemli değişken {onem.index[0]}, ortalama |SHAP| {onem.iloc[0]:.2f}). Tamamlama kodun: " + _k("TUw1LVNIUC05WEtE"))

kredi.head(3)

## Görev A — Boosting

1. `kredi`'yi `train_test_split(kredi, test_size=0.3, random_state=0, stratify=kredi["default"])` ile `egitim` ve `test` olarak böl.
2. Özellikler: `X_e` ve `X_t`, `default`, `sex` ve `marriage` dışındaki bütün sütunlar (ham hâlleriyle).
3. `HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=8, early_stopping=False, random_state=1)` için `max_iter` adaylarını `[100, 200, 400, 800]` **yalnızca eğitim verisinde** `cross_val_score(..., cv=5, scoring="roc_auc")` ortalamasıyla karşılaştır; en iyisi `secilen_n`.
4. Modeli bu değerle eğitim verisinin tamamında kur: `model`. Test olasılıkları `p_test`, test AUC'si `auc_test`. Modül 4'teki lojistik regresyonun 0,762'siyle karşılaştır.

**Denetim soruları:** sex ve marriage dışarıda mı? Ağaç sayısı test verisiyle mi seçildi? `early_stopping=False` mı?

In [ ]:
# Görev A: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol A — çalıştır, değiştirme (yarım dakika kadar sürebilir)

In [ ]:
kontrol_A(egitim, test, X_e, X_t, secilen_n, model, p_test, auc_test)

## Görev B — SHAP

1. `ex = shap.TreeExplainer(model)` ile ilk 2.000 test müşterisinin SHAP değerlerini hesapla: `sv = ex.shap_values(X_t.iloc[:2000])`. Taban değer `taban = float(np.ravel(ex.expected_value)[0])`.
2. Toplama özelliğini sına: `taban + sv.sum(1)` ile `model.decision_function(X_t.iloc[:2000])` arasındaki en büyük fark.
3. `onem`: her sütunun ortalama |SHAP| değeri, sütun adlarıyla, büyükten küçüğe sıralı bir pandas Series.
4. En önemli değişken için `shap.dependence_plot(onem.index[0], sv, X_t.iloc[:2000])` çiz ve bir cümleyle yorumla. İstersen bir müşterinin şelale grafiğini de çiz: `shap.plots.waterfall(ex(X_t.iloc[:2000])[0])`.

**Denetim soruları:** Katkılar hangi ölçekte (log-odds mu, olasılık mı)? Gemini'nin yorumu “model bu değişkene dayanıyor” mu diyor, yoksa “bu değişken temerrüde yol açıyor” mu?

In [ ]:
# Görev B: Gemini ile kodu burada üret, oku, sonra çalıştır.

### Kontrol B — çalıştır, değiştirme

In [ ]:
kontrol_B(model, X_t, sv, taban, onem)

## Denetim notların

- Gemini ağaç sayısını nasıl seçti? Değiştirmen gerekti mi?
- Boosting lojistik regresyona göre ne kazandırdı? Bu fark bir kararı değiştirir mi?
- SHAP sonuçlarını bir kredi komitesine nasıl anlatırsın? (en önemli değişkenler, log-odds ölçeği, nedensellik değil, adillik)